In [134]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from collections import Counter

#https://www.kaggle.com/datasets/davidcariboo/player-scores
df = pd.read_csv("../players.csv")

df['foot_encoded'] = df['foot'].map({'left': 0, 'right': 1})
df['position_encoded'] = df['position'].map({'Goalkeeper': 0, 'Defender': 1, 'Attack': 2, 'Midfield': 3})
data = df[['height_in_cm', 'market_value_in_eur', 'foot_encoded', 'position_encoded']].dropna()
data['high_value'] = (data['market_value_in_eur'] > data['market_value_in_eur'].median()).astype(int)
# log transform market value
data['market_value_in_eur'] = np.log1p(data['market_value_in_eur'])

# remove outliers using IQR
for col in ['height_in_cm', 'market_value_in_eur']:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data = data[(data[col] >= lower_bound) & (data[col] <= upper_bound)]

X = data[['height_in_cm', 'foot_encoded', 'position_encoded']].values
y = data['high_value'].values

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print("Distribution in train:", Counter(y_train))
print("Distribution in test:", Counter(y_test))

majority = max(Counter(y_train).values())
minority = min(Counter(y_train).values())

if minority / majority < 0.3:
    smote = SMOTE()
    X_train, y_train = smote.fit_resample(X_train, y_train)
    print("After smote:", Counter(y_train))

# z-score
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

print(f"Training set size: {X_train_poly.shape[0]}")
print(f"Test set size: {X_test_poly.shape[0]}")

Distribution in train: Counter({np.int64(0): 12231, np.int64(1): 8841})
Distribution in test: Counter({np.int64(0): 3072, np.int64(1): 2197})
Training set size: 21072
Test set size: 5269


In [135]:
class LogisticRegressionImpl:
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        num_samples, num_features = X.shape
        self.weights = np.zeros(num_features)
        self.bias = 0

        for _ in range(self.epochs):
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = self.sigmoid(linear_model)

            dw = (1 / num_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / num_samples) * np.sum(y_predicted - y)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = self.sigmoid(linear_model)
        return [1 if i > 0.5 else 0 for i in y_predicted]

# training..
lr = LogisticRegressionImpl(learning_rate=0.01, epochs=1000)
lr.fit(X_train_poly, y_train)

# predicting with our model
y_pred = lr.predict(X_test_poly)

print("IMpl logistic regresion accuracy:", accuracy_score(y_test, y_pred))

IMpl logistic regresion accuracy: 0.5830328335547542


In [136]:
# compare with scikit-learn
lr_sklearn = LogisticRegression()
lr_sklearn.fit(X_train_poly, y_train)
y_pred_sklearn = lr_sklearn.predict(X_test_poly)

print("scikit-learn logistic reg accuracy:", accuracy_score(y_test, y_pred_sklearn))

print("our impl report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("scikit-learn report:")
print(classification_report(y_test, y_pred_sklearn, zero_division=0))

# check for similarity
similar_predictions = np.sum(np.array(y_pred) == y_pred_sklearn)
total_predictions = len(y_test)
print(f"match between our impl and scikit: {similar_predictions}/{total_predictions} ({similar_predictions/total_predictions*100:.2f}%)")

scikit-learn logistic reg accuracy: 0.5822736762193965
our impl report:
              precision    recall  f1-score   support

           0       0.59      0.97      0.73      3072
           1       0.50      0.04      0.07      2197

    accuracy                           0.58      5269
   macro avg       0.54      0.51      0.40      5269
weighted avg       0.55      0.58      0.45      5269

scikit-learn report:
              precision    recall  f1-score   support

           0       0.59      0.96      0.73      3072
           1       0.49      0.06      0.10      2197

    accuracy                           0.58      5269
   macro avg       0.54      0.51      0.42      5269
weighted avg       0.55      0.58      0.47      5269

match between our impl and scikit: 5165/5269 (98.03%)


In [137]:
lr_scaled = LogisticRegression(class_weight='balanced')
lr_scaled.fit(X_train_scaled, y_train)
y_pred_lr_scaled = lr_scaled.predict(X_test_scaled)

print("LR scaled features accuracy:", accuracy_score(y_test, y_pred_lr_scaled))
print("classification report:")
print(classification_report(y_test, y_pred_lr_scaled, zero_division=0))
print("confusion matrix:")
print(confusion_matrix(y_test, y_pred_lr_scaled))

LR scaled features accuracy: 0.5304611880812299
classification report:
              precision    recall  f1-score   support

           0       0.61      0.55      0.58      3072
           1       0.44      0.51      0.47      2197

    accuracy                           0.53      5269
   macro avg       0.53      0.53      0.52      5269
weighted avg       0.54      0.53      0.53      5269

confusion matrix:
[[1681 1391]
 [1083 1114]]


In [138]:
lr_poly = LogisticRegression(class_weight='balanced')
lr_poly.fit(X_train_poly, y_train)
y_pred_lr_poly = lr_poly.predict(X_test_poly)

print("LR polynomial features accuracy:", accuracy_score(y_test, y_pred_lr_poly))
print("classification report:")
print(classification_report(y_test, y_pred_lr_poly, zero_division=0))
print("confusion matrix:")
print(confusion_matrix(y_test, y_pred_lr_poly))

LR polynomial features accuracy: 0.5243879293983678
classification report:
              precision    recall  f1-score   support

           0       0.62      0.48      0.54      3072
           1       0.45      0.58      0.51      2197

    accuracy                           0.52      5269
   macro avg       0.53      0.53      0.52      5269
weighted avg       0.55      0.52      0.53      5269

confusion matrix:
[[1482 1590]
 [ 916 1281]]


In [139]:
rf_scaled = RandomForestClassifier()
rf_scaled.fit(X_train_scaled, y_train)
y_pred_rf_scaled = rf_scaled.predict(X_test_scaled)

print("RF scaled features accuracy:", accuracy_score(y_test, y_pred_rf_scaled))
print("classification report:")
print(classification_report(y_test, y_pred_rf_scaled, zero_division=0))
print("confusion matrix:")
print(confusion_matrix(y_test, y_pred_rf_scaled))

RF scaled features accuracy: 0.5739229455304612
classification report:
              precision    recall  f1-score   support

           0       0.59      0.88      0.71      3072
           1       0.46      0.14      0.21      2197

    accuracy                           0.57      5269
   macro avg       0.53      0.51      0.46      5269
weighted avg       0.54      0.57      0.50      5269

confusion matrix:
[[2717  355]
 [1890  307]]


In [140]:
rf_poly = RandomForestClassifier()
rf_poly.fit(X_train_poly, y_train)
y_pred_rf_poly = rf_poly.predict(X_test_poly)

print("RF polynomial features accuracy:", accuracy_score(y_test, y_pred_rf_poly))
print("classification report:")
print(classification_report(y_test, y_pred_rf_poly, zero_division=0))
print("confusion matrix:")
print(confusion_matrix(y_test, y_pred_rf_poly))

RF polynomial features accuracy: 0.5767697855380528
classification report:
              precision    recall  f1-score   support

           0       0.59      0.89      0.71      3072
           1       0.47      0.14      0.22      2197

    accuracy                           0.58      5269
   macro avg       0.53      0.51      0.46      5269
weighted avg       0.54      0.58      0.50      5269

confusion matrix:
[[2733  339]
 [1891  306]]


In [141]:
# compare all models
models = ['LR scaled', 'LR poly', 'RF scaled', 'RF poly']
accuracies = [
    accuracy_score(y_test, y_pred_lr_scaled),
    accuracy_score(y_test, y_pred_lr_poly),
    accuracy_score(y_test, y_pred_rf_scaled),
    accuracy_score(y_test, y_pred_rf_poly)
]

for model, acc in zip(models, accuracies):
    print(f"{model}: {acc:.4f}")

best_idx = accuracies.index(max(accuracies))
print(f"Best model: {models[best_idx]} with accuracy {accuracies[best_idx]:.4f}")

LR scaled: 0.5305
LR poly: 0.5244
RF scaled: 0.5739
RF poly: 0.5768
Best model: RF poly with accuracy 0.5768


In [ ]:
import warnings
warnings.filterwarnings('ignore')

param_grid = [
    {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1'], 'solver': ['liblinear'], 'max_iter': [1000]},
    {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l2'], 'solver': ['lbfgs'], 'max_iter': [1000]}
]

grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_poly, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cv score:", grid_search.best_score_)

best_lr = grid_search.best_estimator_
y_pred_tuned = best_lr.predict(X_test_poly)
print("tuned LR accuracy:", accuracy_score(y_test, y_pred_tuned))
print("classification report:")
print(classification_report(y_test, y_pred_tuned, zero_division=0))

y_pred_proba = best_lr.predict_proba(X_test_poly)[:, 1]
y_pred_threshold = (y_pred_proba > 0.4).astype(int)
print("with threshold 0.4 accuracy:", accuracy_score(y_test, y_pred_threshold))
print("classification report:")
print(classification_report(y_test, y_pred_threshold, zero_division=0))

Best parameters: {'C': 0.01, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'lbfgs'}
Best cv score: 0.5802964078952776
tuned LR accuracy: 0.5822736762193965
classification report:
              precision    recall  f1-score   support

           0       0.59      0.96      0.73      3072
           1       0.49      0.06      0.10      2197

    accuracy                           0.58      5269
   macro avg       0.54      0.51      0.42      5269
weighted avg       0.55      0.58      0.47      5269

with threshold 0.4 accuracy: 0.4980072119946859
classification report:
              precision    recall  f1-score   support

           0       0.64      0.32      0.43      3072
           1       0.44      0.75      0.55      2197

    accuracy                           0.50      5269
   macro avg       0.54      0.53      0.49      5269
weighted avg       0.56      0.50      0.48      5269

